In [1]:
from pyrosm import OSM
import pandas as pd

FILES = [
    ("ES", "spain-260607.osm.pbf"),
    ("PT", "portugal-260607.osm.pbf"),
]

PLACE_TYPES = ["city", "town", "village", "hamlet", "locality", "suburb"]

all_places = []

for country, pbf_file in FILES:
    osm = OSM(pbf_file)

    places = osm.get_pois(
        custom_filter={"place": PLACE_TYPES}
    )

    if places is None or places.empty:
        continue

    places = places[places.geometry.type == "Point"].copy()

    places["country"] = country
    places["lon"] = places.geometry.x
    places["lat"] = places.geometry.y

    cols = [
        "country",
        "id",
        "name",
        "place",
        "population",
        "lat",
        "lon",
    ]

    available_cols = [c for c in cols if c in places.columns]
    places = places[available_cols]

    all_places.append(places)

df = pd.concat(all_places, ignore_index=True)

df = df.dropna(subset=["name", "lat", "lon"])
df = df.drop_duplicates(subset=["country", "id"])

df.to_parquet("iberia.parquet", index=False)

print(df.shape)
print(df.head(20))

ModuleNotFoundError: No module named 'pyrosm'

In [2]:
!pip show pyrosm